In [2]:
import pandas as pd
import sqlite3

In [3]:
#Carregando o CSV em um DataFrame
file_path = "london_houses.csv"
df = pd.read_csv(file_path)

#Exibindo as primeiras linhas para verificar
df.head()

,Address,Neighborhood,Bedrooms,Bathrooms,Square Meters,Building Age,Garden,Garage,Floors,Property Type,Heating Type,Balcony,Interior Style,View,Materials,Building Status,Price (£)
0,78 Regent Street,Notting Hill,2,3,179,72,No,No,3,Semi-Detached,Electric Heating,High-level Balcony,Industrial,Garden,Marble,Renovated,2291200
1,198 Oxford Street,Westminster,2,1,123,34,Yes,No,1,Apartment,Central Heating,High-level Balcony,Industrial,City,Laminate Flooring,Old,1476000
2,18 Regent Street,Soho,5,3,168,38,No,Yes,3,Semi-Detached,Central Heating,No Balcony,Industrial,Street,Wood,Renovated,1881600
3,39 Piccadilly Circus,Islington,5,1,237,53,Yes,Yes,1,Apartment,Underfloor Heating,No Balcony,Classic,Park,Granite,Renovated,1896000
4,116 Fleet Street,Marylebone,4,1,127,23,No,Yes,2,Semi-Detached,Central Heating,No Balcony,Modern,Park,Wood,Old,1524000


In [4]:
#Conectando ao banco de dados SQLite (em memória ou em arquivo)
conn = sqlite3.connect(':memory:')  #Usar ':memory:' para um banco em memória ou 'file.db' para um arquivo

#Carregando o DataFrame no SQLite como uma tabela
df.to_sql("london_houses", conn, if_exists="replace", index=False)

1000

In [5]:
#Criando uma nova tabela com o PropertyID auto-incrementado
query = """
CREATE TABLE PropertiesWithID AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY Address) AS PropertyID,
    *
FROM london_houses;
"""
conn.execute(query)

In [6]:
#Consultando a nova tabela
result = pd.read_sql_query("SELECT * FROM PropertiesWithID", conn)

#Consultando a nova tabela
result = pd.read_sql_query("SELECT * FROM PropertiesWithID", conn)

#Exibindo as primeiras linhas
print("Tabela com PropertyID:")
result.head()

Tabela com PropertyID:


,PropertyID,Address,Neighborhood,Bedrooms,Bathrooms,Square Meters,Building Age,Garden,Garage,Floors,Property Type,Heating Type,Balcony,Interior Style,View,Materials,Building Status,Price (£)
0,1,1 Baker Street,Chelsea,2,2,216,74,No,Yes,3,Apartment,Underfloor Heating,Low-level Balcony,Minimalist,Park,Marble,Old,2880000
1,2,1 Regent Street,Shoreditch,1,3,118,16,No,Yes,3,Semi-Detached,Electric Heating,High-level Balcony,Industrial,Park,Marble,New,1038399
2,3,10 Bond Street,Soho,3,3,229,86,Yes,Yes,2,Apartment,Central Heating,High-level Balcony,Minimalist,Park,Marble,New,2137333
3,4,10 Fleet Street,Greenwich,4,2,210,41,Yes,Yes,1,Semi-Detached,Gas Heating,No Balcony,Classic,Street,Wood,New,1680000
4,5,10 Park Lane,Westminster,2,1,173,56,Yes,Yes,1,Detached House,Gas Heating,High-level Balcony,Modern,Park,Laminate Flooring,Renovated,3114000


In [7]:
print(result.columns)

Index(['PropertyID', 'Address', 'Neighborhood', 'Bedrooms', 'Bathrooms',
       'Square Meters', 'Building Age', 'Garden', 'Garage', 'Floors',
       'Property Type', 'Heating Type', 'Balcony', 'Interior Style', 'View',
       'Materials', 'Building Status', 'Price (£)'],
      dtype='object')


In [8]:
#Criando a tabela info_properties
conn.execute("""
CREATE TABLE info_properties AS
SELECT
    PropertyID,
    Address,
    Neighborhood,
    "Price (£)"  as "Price_£",
    "Square Meters" as Square_Meters,
    Bedrooms,
    Bathrooms,
    "Building Age" as "Building_Age",
    Garden,
    Garage,
    Floors
FROM PropertiesWithID;
""")

#Criando a tabela property_features
conn.execute("""
CREATE TABLE property_features AS
SELECT
    PropertyID,
    "Property Type" as Property_Type,
    "Heating Type" as Heating_Type,
    Balcony,
    "Interior Style" as Interior_Style,
    View,
    Materials,
    "Building Status" as Building_Status
FROM PropertiesWithID;
""")

In [9]:
#Verificando as tabelas criadas
# properties = pd.read_sql_query("SELECT * FROM info_properties", conn)
# features = pd.read_sql_query("SELECT * FROM property_features", conn)

# print("\nTabela Properties:")
# print(properties.head())

# print("\nTabela PropertyFeatures:")
# print(features.head())

In [10]:
#Verificando as tabelas criadas
query = pd.read_sql_query( """
SELECT 
    *
FROM info_properties
LIMIT 10;
""", conn)
query

,PropertyID,Address,Neighborhood,Price_£,Square_Meters,Bedrooms,Bathrooms,Building_Age,Garden,Garage,Floors
0,1,1 Baker Street,Chelsea,2880000,216,2,2,74,No,Yes,3
1,2,1 Regent Street,Shoreditch,1038399,118,1,3,16,No,Yes,3
2,3,10 Bond Street,Soho,2137333,229,3,3,86,Yes,Yes,2
3,4,10 Fleet Street,Greenwich,1680000,210,4,2,41,Yes,Yes,1
4,5,10 Park Lane,Westminster,3114000,173,2,1,56,Yes,Yes,1
5,6,10 Piccadilly Circus,Soho,1512000,162,5,3,5,No,Yes,1
6,7,10 Regent Street,Kensington,1535199,101,2,1,46,No,Yes,1
7,8,100 Baker Street,Chelsea,4280000,214,4,3,21,Yes,Yes,2
8,9,100 Bond Street,Islington,608000,76,4,3,27,Yes,No,2
9,10,100 Fleet Street,Camden,530399,51,1,3,59,No,Yes,2


In [11]:
#Verificando as tabelas criadas
query = pd.read_sql_query( """
SELECT 
    *
FROM property_features
LIMIT 10;
""", conn)
query

,PropertyID,Property_Type,Heating_Type,Balcony,Interior_Style,View,Materials,Building_Status
0,1,Apartment,Underfloor Heating,Low-level Balcony,Minimalist,Park,Marble,Old
1,2,Semi-Detached,Electric Heating,High-level Balcony,Industrial,Park,Marble,New
2,3,Apartment,Central Heating,High-level Balcony,Minimalist,Park,Marble,New
3,4,Semi-Detached,Gas Heating,No Balcony,Classic,Street,Wood,New
4,5,Detached House,Gas Heating,High-level Balcony,Modern,Park,Laminate Flooring,Renovated
5,6,Apartment,Electric Heating,Low-level Balcony,Classic,Park,Wood,New
6,7,Semi-Detached,Underfloor Heating,Low-level Balcony,Modern,Street,Laminate Flooring,Old
7,8,Detached House,Underfloor Heating,High-level Balcony,Modern,City,Laminate Flooring,Renovated
8,9,Apartment,Gas Heating,Low-level Balcony,Classic,Park,Wood,New
9,10,Semi-Detached,Electric Heating,Low-level Balcony,Modern,Sea,Marble,Old


# Qual é o preço médio dos imóveis por bairro?

In [13]:
avg_neighborhood = pd.read_sql_query( """
SELECT 
    Neighborhood,
    COUNT (*) as "Qtd_Properties",
    ROUND(AVG (Price_£) / 1000000, 2) as "Avg_Price_(Million)"
FROM info_properties
GROUP BY Neighborhood

""", conn)
avg_neighborhood

,Neighborhood,Qtd_Properties,Avg_Price_(Million)
0,Camden,106,1.63
1,Chelsea,94,2.45
2,Greenwich,97,1.27
3,Islington,97,1.51
4,Kensington,114,2.28
5,Marylebone,113,1.82
6,Notting Hill,96,1.96
7,Shoreditch,89,1.33
8,Soho,96,1.78
9,Westminster,98,2.29


A região de Kensington tem a maior concentração de imoveis a venda e apresenta a terceira maior média de preço, ficando atrás de Chelsea e Westminster.

# Quais são os imóveis com mais de 3 quartos e mais de 2 banheiros?

In [16]:
properties_3q_2b = pd.read_sql_query( """
SELECT 
    *
FROM info_properties
WHERE Bedrooms >= 3 AND	Bathrooms >= 2

""", conn)
properties_3q_2b

,PropertyID,Address,Neighborhood,Price_£,Square_Meters,Bedrooms,Bathrooms,Building_Age,Garden,Garage,Floors
0,3,10 Bond Street,Soho,2137333,229,3,3,86,Yes,Yes,2
1,4,10 Fleet Street,Greenwich,1680000,210,4,2,41,Yes,Yes,1
2,6,10 Piccadilly Circus,Soho,1512000,162,5,3,5,No,Yes,1
3,8,100 Baker Street,Chelsea,4280000,214,4,3,21,Yes,Yes,2
4,9,100 Bond Street,Islington,608000,76,4,3,27,Yes,No,2
...,...,...,...,...,...,...,...,...,...,...,...
406,991,98 Camden High Street,Marylebone,2000000,200,3,2,71,Yes,Yes,2
407,992,98 Fleet Street,Marylebone,2280000,152,3,2,44,No,No,3
408,993,98 Strand,Kensington,1735333,137,5,2,8,Yes,No,1
409,995,99 Bond Street,Kensington,3389599,223,4,2,26,Yes,No,1


# Quais imóveis possuem garagem e jardim ao mesmo tempo?

In [18]:
properties_garden_garage = pd.read_sql_query( """
SELECT 
    *
FROM info_properties
WHERE Garden = "Yes" 
AND	Garage = "Yes"

""", conn)
properties_garden_garage

,PropertyID,Address,Neighborhood,Price_£,Square_Meters,Bedrooms,Bathrooms,Building_Age,Garden,Garage,Floors
0,3,10 Bond Street,Soho,2137333,229,3,3,86,Yes,Yes,2
1,4,10 Fleet Street,Greenwich,1680000,210,4,2,41,Yes,Yes,1
2,5,10 Park Lane,Westminster,3114000,173,2,1,56,Yes,Yes,1
3,8,100 Baker Street,Chelsea,4280000,214,4,3,21,Yes,Yes,2
4,17,102 Bond Street,Shoreditch,1540000,210,3,1,74,Yes,Yes,2
...,...,...,...,...,...,...,...,...,...,...,...
252,989,97 Regent Street,Greenwich,1550000,155,3,2,33,Yes,Yes,3
253,991,98 Camden High Street,Marylebone,2000000,200,3,2,71,Yes,Yes,2
254,994,99 Bond Street,Greenwich,1013333,152,1,3,52,Yes,Yes,3
255,998,99 Oxford Street,Kensington,2234400,147,4,1,83,Yes,Yes,3


# Qual bairro tem o maior número de imóveis renovados?

In [20]:
properties_renovated = pd.read_sql_query( """
SELECT 
    ip.Neighborhood, 
    count (*) AS Qtd_renovated
FROM info_properties ip
INNER JOIN property_features fp on ip.PropertyID = fp.PropertyID
WHERE fp.Building_Status = "Renovated"
GROUP BY ip.Neighborhood
ORDER BY qtd_renovated desc

""", conn)
properties_renovated 

,Neighborhood,Qtd_renovated
0,Westminster,41
1,Kensington,39
2,Soho,38
3,Chelsea,36
4,Islington,35
5,Camden,34
6,Greenwich,33
7,Shoreditch,31
8,Marylebone,31
9,Notting Hill,23


# Quais tipos de aquecimento são mais comuns para imóveis com varanda?

In [22]:
properties_heat_balcony = pd.read_sql_query( """
SELECT 
    Heating_Type,	 
    count (*) AS Qtd
FROM property_features
WHERE Balcony not in ("No Balcony")
GROUP BY Heating_Type	
ORDER BY Qtd desc

""", conn)
properties_heat_balcony

,Heating_Type,Qtd
0,Gas Heating,176
1,Electric Heating,173
2,Underfloor Heating,159
3,Central Heating,152


# Quais imóveis têm vista para o mar e preço acima de £2.000.000?

In [24]:
view_sea = pd.read_sql_query( """
SELECT 
    ip.PropertyID,	
    ip.Address,	
    ip.Neighborhood,	
    ip.Price_£, 
    fp.View
FROM info_properties ip
INNER JOIN property_features fp on ip.PropertyID = fp.PropertyID
WHERE fp.View = "Sea"
AND ip.Price_£ >=2000000
--GROUP BY ip.Neighborhood
ORDER BY ip.Price_£ desc

""", conn)
view_sea

,PropertyID,Address,Neighborhood,Price_£,View
0,872,77 Oxford Street,Chelsea,4980000,Sea
1,280,150 Oxford Street,Kensington,4427000,Sea
2,581,22 Park Lane,Westminster,4320000,Sea
3,74,112 Park Lane,Kensington,4066000,Sea
4,693,42 Oxford Street,Chelsea,3952000,Sea
...,...,...,...,...,...
77,289,152 Regent Street,Marylebone,2088000,Sea
78,72,112 Fleet Street,Camden,2079999,Sea
79,792,60 Park Lane,Camden,2069599,Sea
80,374,169 King's Road,Camden,2059199,Sea


# Qual a distribuição do número de andares dos imóveis em cada bairro?

In [26]:
floors_neighborhood = pd.read_sql_query( """
SELECT 
   Neighborhood,
   Floors,
   count (Floors) AS Qtd
FROM info_properties
GROUP BY Neighborhood, Floors
ORDER BY Qtd desc

""", conn)
floors_neighborhood

,Neighborhood,Floors,Qtd
0,Marylebone,1,46
1,Kensington,1,43
2,Chelsea,3,39
3,Camden,1,37
4,Greenwich,2,37
5,Westminster,3,37
6,Camden,2,36
7,Kensington,2,36
8,Notting Hill,3,36
9,Islington,2,35


O bairro de Marylebone apresenta maior quantidade de imóveis com apensas um andar. 
Em imóveis de 2 andares temos Greenwich e encontramos 39 imóveis de 3 andares em Chelsea.

# Quais bairros têm a maior proporção de casas com aquecimento elétrico?

In [29]:
neighborhood_heating = pd.read_sql_query( """

WITH 
neighborhood as (
        SELECT 
            ip.Neighborhood,
            count (*) AS qtd_total
        FROM info_properties ip
        INNER JOIN property_features fp on ip.PropertyID = fp.PropertyID
        WHERE fp.Property_Type in ("Detached House", "Semi-Detached")
        GROUP BY ip.Neighborhood
       ),
        
neighborhood_heating as (
        SELECT 
            ip.Neighborhood, 
            count (*) AS qtd_eletric
        FROM info_properties ip
        INNER JOIN property_features fp on ip.PropertyID = fp.PropertyID
        WHERE fp.Property_Type in ("Detached House", "Semi-Detached")
        AND fp.Heating_Type in ("Electric Heating")
        GROUP BY ip.Neighborhood
        )

SELECT 
    neh.Neighborhood, 
    ROUND(CAST(neh.qtd_eletric AS FLOAT) / CAST(nh.qtd_total AS FLOAT)*100, 2) AS "Proportion (%)"
FROM neighborhood_heating neh
JOIN neighborhood nh on neh.Neighborhood = nh.Neighborhood
ORDER BY "Proportion (%)" desc

""", conn)
neighborhood_heating

,Neighborhood,Proportion (%)
0,Greenwich,33.33
1,Marylebone,30.67
2,Notting Hill,29.41
3,Soho,28.99
4,Chelsea,25.71
5,Islington,24.29
6,Shoreditch,24.07
7,Kensington,23.68
8,Westminster,21.43
9,Camden,18.42


# Qual a distribução de status de imóveis e idade por bairro?

In [31]:
properties_status = pd.read_sql_query("""
WITH
age_range AS (
    SELECT 
        ip.Neighborhood,
        fp.Building_Status,
        COUNT(*) AS Qtd_Properties,
        ROUND(AVG(ip.Building_Age),0) AS Average_Age,
        MAX(ip.Building_Age) AS Oldest_Age,
        MIN(ip.Building_Age) AS Younger_Age,
        ROUND(AVG(ip.Price_£) / 1000000, 2) AS Avg_Price_Million
    FROM info_properties ip
    INNER JOIN property_features fp ON ip.PropertyID = fp.PropertyID  
    GROUP BY 1, 2
),

neighborhood AS (
    SELECT 
        Neighborhood,
        COUNT(*) AS Qtd_Properties        
    FROM info_properties
    GROUP BY Neighborhood
)
SELECT ar.Neighborhood, 
    ar.Building_Status, 
    ar.Qtd_Properties, 
    ROUND(CAST(ar.Qtd_Properties AS FLOAT) / CAST(nh.Qtd_Properties AS FLOAT) * 100, 2) AS "Proportion (%)",
    ar.Average_Age,
    ar.Oldest_Age,
    ar.Younger_Age,
    ar.Avg_Price_Million
FROM age_range ar
JOIN neighborhood nh ON ar.Neighborhood = nh.Neighborhood
""", conn)

properties_status

,Neighborhood,Building_Status,Qtd_Properties,Proportion (%),Average_Age,Oldest_Age,Younger_Age,Avg_Price_Million
0,Camden,New,40,37.74,49.0,99,3,1.58
1,Camden,Old,32,30.19,49.0,94,2,1.61
2,Camden,Renovated,34,32.08,45.0,89,2,1.70
3,Chelsea,New,27,28.72,47.0,91,6,2.53
4,Chelsea,Old,31,32.98,49.0,99,2,2.30
5,Chelsea,Renovated,36,38.30,56.0,99,1,2.53
6,Greenwich,New,30,30.93,41.0,82,2,1.21
7,Greenwich,Old,34,35.05,54.0,98,2,1.20
8,Greenwich,Renovated,33,34.02,40.0,97,2,1.40
9,Islington,New,37,38.14,53.0,95,2,1.59


Aqui observamos que a base de dados pode estar com registros incorretos ou inconsistentes, visto que temos imóveis classificados como "Novos" porém a idade da construção indica que ela foi construída há mais de 90 anos. Isso é contraditório, pois um imóvel classificado como "Novo" deveria ter uma idade de construção muito mais recente.

# Buscando imóveis em Notting Hill

In [34]:
notting_hill = pd.read_sql_query( """

SELECT 
ip.Neighborhood,
    COUNT(ip.PropertyID) AS "Total Properties",
    ROUND(AVG (ip.Price_£) / 1000000, 2) AS "Average Price (£) - Million",
    ROUND(AVG(ip.Square_Meters),2) AS "Average Area (m²)",
    ROUND(AVG(ip.Bedrooms),0) AS "Average Number of Bedrooms",
    ROUND(AVG(ip.Bathrooms),0) AS "Average Number of Bathrooms",
    SUM(CASE WHEN ip.Garden = 'Yes' THEN 1 ELSE 0 END) AS "Properties with Garden",
    SUM(CASE WHEN ip.Garage = 'Yes' THEN 1 ELSE 0 END) AS "Properties with Garage",
    SUM(CASE WHEN fp.Heating_Type = 'Electric Heating' THEN 1 ELSE 0 END) AS "Properties with Electric Heating",
    SUM(CASE WHEN fp.Balcony = 'No Balcony' THEN 0 ELSE 1 END) AS "Properties with Balcony",
    SUM(CASE WHEN fp.View = 'Park' THEN 1 ELSE 0 END) AS "Park Views",
    ROUND(AVG(ip.Building_Age),0) AS "Average Property Age"
FROM info_properties ip
INNER JOIN property_features fp on ip.PropertyID = fp.PropertyID    
WHERE ip.Neighborhood in ("Notting Hill")
AND fp.Materials in ("Wood")
GROUP BY  Neighborhood

""", conn)

notting_hill

,Neighborhood,Total Properties,Average Price (£) - Million,Average Area (m²),Average Number of Bedrooms,Average Number of Bathrooms,Properties with Garden,Properties with Garage,Properties with Electric Heating,Properties with Balcony,Park Views,Average Property Age
0,Notting Hill,24,2.28,160.54,3.0,2.0,13,15,6,18,3,53.0
